# ✈️ Behind the Boarding Pass

## A Study in Passenger Behavior Analytics, Explainable AI & Machine Learning

---

> **Dataset:** Kaggle Spaceship Titanic Dataset

> **Author:** Hifza Amir

> **Focus Areas:** Data Science, Behavioral Analytics, Explainable AI, and Machine Learning

---

### Project Abstract

Every passenger carries a unique story hidden within their travel patterns, spending behavior, demographics, and group dynamics. This project explores those hidden patterns to understand what factors influence passenger transportation outcomes.

Rather than approaching the dataset as a simple prediction problem, this project investigates passenger behavior through data analysis, feature engineering, explainable machine learning, and risk profiling. By combining behavioral insights with predictive modeling, the project aims to uncover meaningful relationships between passenger characteristics and transportation outcomes.

**Key highlights of this project:**

1. Advanced feature engineering from passenger identifiers, cabin information, and expenditure patterns
2. Behavioral analytics focused on spending habits, travel groups, and passenger profiles
3. Leak-free machine learning pipeline with robust preprocessing
4. Hyperparameter optimization using Optuna
5. Explainable AI through SHAP-based global and individual prediction explanations
6. Passenger risk profiling using probability-based segmentation
7. Anomaly detection and behavioral pattern analysis
8. Ensemble learning using multiple machine learning models for reliable predictions



## ⚙️ Environment Setup

In [1]:
# Install additional packages if needed
# !pip install missingno shap optuna lightgbm xgboost catboost plotly -q
import warnings
warnings.filterwarnings("ignore")


## 📦 Imports

In [2]:
# ── Standard library ──────────────────────────────────────────────────────────
import os, re, json
from pathlib import Path

# ── Data & Math ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from scipy import stats

# ── Visualization ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import missingno as msno

# ── ML — Preprocessing ────────────────────────────────────────────────────────
from sklearn.pipeline          import Pipeline
from sklearn.compose           import ColumnTransformer
from sklearn.impute            import SimpleImputer
from sklearn.preprocessing     import StandardScaler, LabelEncoder, OrdinalEncoder
from sklearn.preprocessing     import OneHotEncoder

# ── ML — Models ───────────────────────────────────────────────────────────────
from sklearn.linear_model      import LogisticRegression
from sklearn.neighbors         import KNeighborsClassifier
from sklearn.svm               import SVC
from sklearn.naive_bayes       import GaussianNB
from sklearn.tree              import DecisionTreeClassifier
from sklearn.ensemble          import (RandomForestClassifier,
                                       GradientBoostingClassifier,
                                       AdaBoostClassifier,
                                       StackingClassifier,
                                       VotingClassifier)
from xgboost                   import XGBClassifier
from catboost                  import CatBoostClassifier
from lightgbm                  import LGBMClassifier

# ── ML — Evaluation & Selection ───────────────────────────────────────────────
from sklearn.model_selection   import (train_test_split,
                                       StratifiedKFold,
                                       cross_val_score,
                                       learning_curve)
from sklearn.metrics           import (accuracy_score, f1_score,
                                       recall_score, precision_score,
                                       confusion_matrix, roc_auc_score,
                                       roc_curve, classification_report)

# ── Explainability ────────────────────────────────────────────────────────────
import shap

# ── Hyperparameter Optimisation ───────────────────────────────────────────────
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Display config ────────────────────────────────────────────────────────────
%matplotlib inline
sns.set_theme(style="darkgrid", font_scale=1.3)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 80)
SEED = 42
np.random.seed(SEED)

PALETTE = ["#5B8DB8", "#E8825A"]   # blue = False, orange = True
print("✅ All imports successful.")


ModuleNotFoundError: No module named 'numpy'

## 📂 Data Loading

> **Note:** Place `train.csv` and `test.csv` (downloaded from Kaggle) in the same directory as this notebook, or update the paths below.


In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
TRAIN_PATH = "train.csv"
TEST_PATH  = "test.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

# Keep a copy of PassengerId for final submission before any transforms
submission_ids = test_df["PassengerId"].copy()

print(f"Train shape : {train_df.shape}")
print(f"Test  shape : {test_df.shape}")
train_df.head(3)


## 📖 Feature Dictionary

| Feature | Type | Description |
|---|---|---|
| `PassengerId` | String | `gggg_pp` — group ID + member number |
| `HomePlanet` | Categorical | Departure planet (Earth, Europa, Mars) |
| `CryoSleep` | Boolean | Whether passenger was in suspended animation |
| `Cabin` | String | `deck/num/side` — ship cabin address |
| `Destination` | Categorical | Arrival planet |
| `Age` | Numeric | Passenger age |
| `VIP` | Boolean | Paid VIP service |
| `RoomService` | Numeric | Spending at Room Service (credits) |
| `FoodCourt` | Numeric | Spending at Food Court |
| `ShoppingMall` | Numeric | Spending at Shopping Mall |
| `Spa` | Numeric | Spending at Spa |
| `VRDeck` | Numeric | Spending at VR Deck |
| `Name` | String | Passenger name (high cardinality — dropped) |
| `Transported` | Boolean | **TARGET** — whether passenger was teleported |


## 🔍 Data Quality Audit

In [ ]:
def quality_audit(df, label):
    print(f"{'═'*55}")
    print(f"  {label}")
    print(f"{'═'*55}")
    print(f"  Rows: {df.shape[0]:,}   Columns: {df.shape[1]}")
    print(f"  Duplicates: {df.duplicated().sum()}")
    null_pct = (df.isnull().sum() / len(df) * 100).round(2)
    null_pct = null_pct[null_pct > 0].sort_values(ascending=False)
    if len(null_pct):
        print("\n  Missing Values (%):")
        for col, pct in null_pct.items():
            bar = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))
            print(f"    {col:<18} {bar}  {pct:.1f}%")
    print()

quality_audit(train_df, "TRAIN DATASET")
quality_audit(test_df,  "TEST  DATASET")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
msno.matrix(train_df, ax=axes[0], sparkline=False, color=(0.36, 0.55, 0.72))
axes[0].set_title("Train — Missing Value Pattern", fontsize=14, fontweight="bold")
msno.bar(train_df, ax=axes[1], color=(0.91, 0.51, 0.35))
axes[1].set_title("Train — Missing Value Count", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## 📊 Exploratory Data Analysis

### 🎯 Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie
counts = train_df["Transported"].value_counts()
axes[0].pie(counts, labels=["Not Transported\n(False)", "Transported\n(True)"],
            autopct="%1.1f%%", colors=PALETTE,
            textprops={"fontsize": 13, "fontweight": "bold"},
            startangle=90, wedgeprops={"edgecolor": "white", "linewidth": 2})
axes[0].set_title("Target Distribution", fontweight="bold", fontsize=14)

# Bar with counts
bars = axes[1].bar(["Not Transported", "Transported"], counts.values,
                   color=PALETTE, edgecolor="white", linewidth=1.5, width=0.5)
for bar, val in zip(bars, counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 f"{val:,}", ha="center", va="bottom", fontweight="bold", fontsize=12)
axes[1].set_title("Passenger Count by Fate", fontweight="bold", fontsize=14)
axes[1].set_ylabel("Count")
axes[1].set_ylim(0, counts.max() * 1.15)
plt.tight_layout()
plt.show()
print(f"Class balance — False: {counts[False]/len(train_df)*100:.1f}%  True: {counts[True]/len(train_df)*100:.1f}%")


### 🌍 Categorical Features vs. Transportation

In [ ]:
cat_cols = ["HomePlanet", "CryoSleep", "Destination", "VIP"]
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, col in zip(axes.flatten(), cat_cols):
    order = train_df[col].value_counts().index.tolist()
    g = sns.countplot(data=train_df, x=col, hue="Transported",
                      order=order, palette=PALETTE, ax=ax,
                      edgecolor="white", linewidth=1)
    ax.set_title(f"{col} vs Transported", fontweight="bold", fontsize=13)
    ax.set_xlabel("")
    ax.legend(title="Transported", labels=["False", "True"])
    
    # Add transport rate labels
    totals = train_df[col].value_counts()
    transported = train_df[train_df["Transported"]==True][col].value_counts()
    for cat in order:
        rate = transported.get(cat, 0) / totals.get(cat, 1) * 100
        x_pos = order.index(cat)
        ax.text(x_pos, ax.get_ylim()[1] * 0.97, f"{rate:.0f}%", 
                ha="center", fontsize=10, color="dimgray", fontweight="bold")

plt.suptitle("Categorical Features — Transport Rate (% shown above bars)", 
             fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


### 💸 Spending Behavior Analysis

In [ ]:
exp_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

fig, axes = plt.subplots(2, 3, figsize=(20, 10))
axes = axes.flatten()

for idx, col in enumerate(exp_cols):
    ax = axes[idx]
    for val, color, label in [(True, PALETTE[1], "Transported"), (False, PALETTE[0], "Not Transported")]:
        subset = train_df[train_df["Transported"] == val][col].dropna()
        subset_log = np.log1p(subset)
        ax.hist(subset_log, bins=50, alpha=0.65, color=color, label=label, edgecolor="none")
    ax.set_title(f"log(1+{col})", fontweight="bold")
    ax.set_xlabel("log-transformed value")
    ax.legend(fontsize=9)

# Correlation heatmap of spending features
ax = axes[5]
spend_corr = train_df[exp_cols + ["Transported"]].copy()
spend_corr["Transported"] = spend_corr["Transported"].astype(int)
corr = spend_corr.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            mask=mask, ax=ax, square=True, linewidths=0.5)
ax.set_title("Spending + Target Correlation", fontweight="bold")

plt.suptitle("Expenditure Feature Distributions (log-scale) & Correlations",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


### 🧬 Age Distribution by Fate

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Distribution
sns.histplot(data=train_df, x="Age", hue="Transported", bins=40,
             kde=True, palette=PALETTE, ax=axes[0], alpha=0.6)
axes[0].set_title("Age Distribution by Transported", fontweight="bold")

# Box
sns.boxplot(data=train_df, x="Transported", y="Age", palette=PALETTE,
            ax=axes[1], width=0.4)
axes[1].set_title("Age Distribution — Box Plot", fontweight="bold")
axes[1].set_xticklabels(["Not Transported", "Transported"])

# Violin
sns.violinplot(data=train_df, x="Transported", y="Age", palette=PALETTE,
               ax=axes[2], inner="quartile", linewidth=1)
axes[2].set_title("Age Distribution — Violin Plot", fontweight="bold")
axes[2].set_xticklabels(["Not Transported", "Transported"])

# Statistical test
g0 = train_df[train_df["Transported"]==False]["Age"].dropna()
g1 = train_df[train_df["Transported"]==True]["Age"].dropna()
t_stat, p_val = stats.mannwhitneyu(g0, g1)
axes[0].text(0.98, 0.95, f"Mann-Whitney U\np = {p_val:.4f}", 
             transform=axes[0].transAxes, ha="right", va="top",
             bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8), fontsize=10)

plt.suptitle("Passenger Age Analysis", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


## 🛠️ Feature Engineering

This section extracts structured information from raw fields and creates domain-informed features. All transformations are implemented as reusable functions that apply identically to train and test sets.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. PassengerId decomposition → Group dynamics
# ─────────────────────────────────────────────────────────────────────────────
def engineer_passenger_id(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["Group"]  = df["PassengerId"].str.split("_").str[0]
    df["Member"] = df["PassengerId"].str.split("_").str[1].astype(int)
    
    group_sizes = df.groupby("Group")["Member"].transform("count")
    df["Group_Size"]      = group_sizes
    df["Travelling_Solo"] = (group_sizes == 1).astype(int)
    df["Is_Group_Leader"] = (df["Member"] == 1).astype(int)
    
    # Drop intermediates
    df.drop(columns=["Group", "Member"], inplace=True)
    return df

# ─────────────────────────────────────────────────────────────────────────────
# 2. Cabin decomposition + regional binning
# ─────────────────────────────────────────────────────────────────────────────
def engineer_cabin(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    cabin_split = df["Cabin"].str.split("/", expand=True)
    df["Cabin_Deck"]   = cabin_split[0]
    df["Cabin_Number"] = pd.to_numeric(cabin_split[1], errors="coerce")
    df["Cabin_Side"]   = cabin_split[2]
    
    # Impute cabin sub-fields
    df["Cabin_Deck"].fillna(df["Cabin_Deck"].mode()[0], inplace=True)
    df["Cabin_Side"].fillna(df["Cabin_Side"].mode()[0], inplace=True)
    df["Cabin_Number"].fillna(df["Cabin_Number"].median(), inplace=True)
    df["Cabin_Number"] = df["Cabin_Number"].astype(int)
    
    # Regional binning (300-unit intervals) 
    df["Cabin_Region"] = pd.cut(df["Cabin_Number"],
                                bins=[-1, 300, 600, 900, 1200, 1500, 10000],
                                labels=[1, 2, 3, 4, 5, 6]).astype(int)
    
    # Deck tier (luxury vs economy — informed by EDA)
    luxury_decks = ["A", "B", "C", "T"]
    df["Is_Luxury_Deck"] = df["Cabin_Deck"].isin(luxury_decks).astype(int)
    
    df.drop(columns=["Cabin_Number"], inplace=True)
    return df

# ─────────────────────────────────────────────────────────────────────────────
# 3. Age group bucketing
# ─────────────────────────────────────────────────────────────────────────────
def engineer_age(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["Age_Group"] = pd.cut(df["Age"],
                             bins=[-1, 12, 18, 25, 32, 50, 200],
                             labels=["Child", "Teen", "YoungAdult",
                                     "Adult", "MidAge", "Senior"])
    df["Is_Minor"] = (df["Age"] <= 18).astype(int)
    return df

# ─────────────────────────────────────────────────────────────────────────────
# 4. Spending behavior features  
# ─────────────────────────────────────────────────────────────────────────────
EXP_COLS = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

def engineer_spending(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["Total_Expenditure"] = df[EXP_COLS].sum(axis=1)
    df["Num_Services_Used"] = (df[EXP_COLS] > 0).sum(axis=1)
    df["No_Spending"]       = (df["Total_Expenditure"] == 0).astype(int)
    
    # Spending diversity — entropy across services
    def spend_entropy(row):
        vals = row[EXP_COLS].values.astype(float)
        total = vals.sum()
        if total == 0:
            return 0.0
        probs = vals / total
        probs = probs[probs > 0]
        return float(-np.sum(probs * np.log(probs + 1e-9)))
    df["Spend_Entropy"] = df.apply(spend_entropy, axis=1)
    
    # Expenditure category
    q33 = df[df["Total_Expenditure"] > 0]["Total_Expenditure"].quantile(0.33)
    q67 = df[df["Total_Expenditure"] > 0]["Total_Expenditure"].quantile(0.67)
    conditions = [
        df["Total_Expenditure"] == 0,
        (df["Total_Expenditure"] > 0) & (df["Total_Expenditure"] <= q33),
        (df["Total_Expenditure"] > q33) & (df["Total_Expenditure"] <= q67),
        df["Total_Expenditure"] > q67
    ]
    df["Spend_Category"] = np.select(conditions,
                                     ["None", "Low", "Medium", "High"],
                                     default="None")
    
    # Dominant spend channel
    def dominant_channel(row):
        vals = row[EXP_COLS]
        if vals.sum() == 0:
            return "None"
        return vals.idxmax()
    df["Dominant_Spend"] = df.apply(dominant_channel, axis=1)
    
    return df

# ─────────────────────────────────────────────────────────────────────────────
# 5. Interaction features (domain-motivated)
# ─────────────────────────────────────────────────────────────────────────────
def engineer_interactions(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # CryoSleep passengers should have 0 spending — flag violations
    cryo = df["CryoSleep"].map({True: 1, False: 0, "True": 1, "False": 0, 1: 1, 0: 0}).fillna(0)
    df["CryoSleep_SpendViolation"] = ((cryo == 1) & (df["Total_Expenditure"] > 0)).astype(int)
    
    # Luxury deck + high spending = high-status passenger
    df["HighStatus"] = ((df["Is_Luxury_Deck"] == 1) & 
                        (df["Spend_Category"] == "High")).astype(int)
    
    # Solo traveller + no spending (reclusive profile)
    df["Reclusive"] = ((df["Travelling_Solo"] == 1) & 
                       (df["No_Spending"] == 1)).astype(int)
    
    # Group traveller who spends a lot (social spender)
    df["Social_Spender"] = ((df["Travelling_Solo"] == 0) & 
                            (df["Spend_Category"].isin(["Medium", "High"]))).astype(int)
    return df

print("✅ Feature engineering functions defined.")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Apply all engineering steps
# ─────────────────────────────────────────────────────────────────────────────
def full_feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    df = engineer_passenger_id(df)
    df = engineer_cabin(df)
    df = engineer_age(df)
    df = engineer_spending(df)
    df = engineer_interactions(df)
    return df

train_fe = full_feature_engineering(train_df.copy())
test_fe  = full_feature_engineering(test_df.copy())

# Drop columns that won't be used in modelling
DROP_COLS = ["PassengerId", "Cabin", "Name"]
train_fe.drop(columns=DROP_COLS, inplace=True)
test_fe.drop(columns=[c for c in DROP_COLS if c in test_fe.columns], inplace=True)

print(f"Train after engineering: {train_fe.shape}")
print(f"Test  after engineering: {test_fe.shape}")
print("\nNew features:")
orig_cols = set(train_df.columns) - set(DROP_COLS)
new_cols  = set(train_fe.columns) - orig_cols
for c in sorted(new_cols):
    print(f"  + {c}")


## 🧠 Behavioral Segmentation

Before modelling, we create passenger spending archetypes — a form of unsupervised behavioral analytics that adds interpretability to our downstream predictions.


In [ ]:
# Visualise spending archetypes (5 channels)
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Spider/radar chart of average spending per transport group
categories = EXP_COLS
N = len(categories)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

ax = axes[0]
ax = plt.subplot(121, polar=True)
for transported, color, label in [(True, PALETTE[1], "Transported"), (False, PALETTE[0], "Not Transported")]:
    values = (train_fe[train_fe["Transported"] == transported][EXP_COLS]
              .median().values.tolist())
    values += values[:1]
    ax.plot(angles, values, "o-", linewidth=2, color=color, label=label)
    ax.fill(angles, values, alpha=0.15, color=color)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)
ax.set_title("Median Spending Profile\nby Passenger Fate", 
             fontweight="bold", pad=20, fontsize=13)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))

# Spending entropy distribution
ax2 = axes[1]
for transported, color, label in [(True, PALETTE[1], "Transported"), (False, PALETTE[0], "Not Transported")]:
    subset = train_fe[train_fe["Transported"] == transported]["Spend_Entropy"].dropna()
    ax2.hist(subset, bins=40, alpha=0.65, color=color, label=label, edgecolor="none")
ax2.set_title("Spending Diversity (Entropy)\nby Passenger Fate", fontweight="bold", fontsize=13)
ax2.set_xlabel("Shannon Entropy across 5 spending channels")
ax2.set_ylabel("Count")
ax2.legend()

plt.tight_layout()
plt.show()


In [ ]:
# Spending category breakdown  
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.countplot(data=train_fe, x="Spend_Category", hue="Transported",
              palette=PALETTE, ax=axes[0],
              order=["None", "Low", "Medium", "High"])
axes[0].set_title("Expenditure Category vs Transported", fontweight="bold")
axes[0].legend(title="Transported")

# Group size analysis
sns.countplot(data=train_fe, x="Group_Size", hue="Transported",
              palette=PALETTE, ax=axes[1])
axes[1].set_title("Group Size vs Transported", fontweight="bold")
axes[1].legend(title="Transported")

plt.tight_layout()
plt.show()


In [ ]:
# Transport rate heatmap: HomePlanet × Destination
pivot = (train_fe.groupby(["HomePlanet", "Destination"])["Transported"]
         .apply(lambda x: (x == True).mean() * 100)
         .unstack(fill_value=0))

plt.figure(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="RdYlGn",
            linewidths=0.5, cbar_kws={"label": "Transport Rate (%)"})
plt.title("Transport Rate (%) — HomePlanet × Destination", 
          fontweight="bold", fontsize=14)
plt.xlabel("Destination")
plt.ylabel("Home Planet")
plt.tight_layout()
plt.show()
print("Insight: Route matters. Some planet pairs show dramatically different transport rates.")


## 🔧 Preprocessing Pipeline (Leak-Free)

A proper `sklearn.Pipeline` ensures:
- Imputation and scaling are **fit only on training data**
- Test data receives `.transform()` — never `.fit_transform()`
- No information from the test set leaks into our model


In [ ]:
# ─── Define column groups ────────────────────────────────────────────────────
TARGET = "Transported"

# Remove target before building column lists
feature_df = train_fe.drop(columns=[TARGET])

# Numeric
NUM_COLS = feature_df.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Boolean/binary that need encoding
BOOL_COLS = ["CryoSleep", "VIP"]  # still object in some cases

# Low-cardinality categoricals for OHE
OHE_COLS = ["HomePlanet", "Destination", "Cabin_Deck", "Cabin_Side", 
            "Age_Group", "Spend_Category", "Dominant_Spend"]

# Ordinal that can be label-encoded
ORD_COLS = ["Cabin_Region"]

print("Numeric cols  :", len(NUM_COLS))
print("OHE cols      :", len(OHE_COLS))
print("Ordinal cols  :", len(ORD_COLS))
print("Bool cols     :", BOOL_COLS)


In [ ]:
# ─── Build ColumnTransformer ──────────────────────────────────────────────────
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler())
])

bool_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder())
])

ohe_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe",     OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

ord_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder())
])

preprocessor = ColumnTransformer(transformers=[
    ("num",  numeric_transformer, NUM_COLS),
    ("bool", bool_transformer,    BOOL_COLS),
    ("ohe",  ohe_transformer,     OHE_COLS),
    ("ord",  ord_transformer,     ORD_COLS),
], remainder="drop")

# ─── Target & split ───────────────────────────────────────────────────────────
X = feature_df
y = train_fe[TARGET].astype(int)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

# Fit preprocessor on train only
X_train_pp = preprocessor.fit_transform(X_train)
X_val_pp   = preprocessor.transform(X_val)
X_test_pp  = preprocessor.transform(test_fe)

print(f"X_train preprocessed: {X_train_pp.shape}")
print(f"X_val   preprocessed: {X_val_pp.shape}")
print(f"X_test  preprocessed: {X_test_pp.shape}")
print("✅ No data leakage — preprocessor fitted on train only.")


## 🤖 Model Benchmarking

We evaluate a broad family of classifiers using **5-fold stratified cross-validation** on the training set, then measure final performance on the held-out validation set.


In [ ]:
# ─── Cross-validation benchmark ──────────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

MODELS = {
    "Logistic Regression":   LogisticRegression(max_iter=1000, random_state=SEED),
    "K-Nearest Neighbors":   KNeighborsClassifier(n_neighbors=11),
    "SVM (RBF)":             SVC(probability=True, random_state=SEED),
    "Naive Bayes":           GaussianNB(),
    "Decision Tree":         DecisionTreeClassifier(max_depth=8, random_state=SEED),
    "Random Forest":         RandomForestClassifier(n_estimators=300, random_state=SEED),
    "AdaBoost":              AdaBoostClassifier(n_estimators=200, random_state=SEED),
    "Gradient Boosting":     GradientBoostingClassifier(n_estimators=300, random_state=SEED),
    "LightGBM":              LGBMClassifier(n_estimators=500, random_state=SEED, verbose=-1),
    "XGBoost":               XGBClassifier(n_estimators=500, use_label_encoder=False,
                                           eval_metric="logloss", random_state=SEED, verbosity=0),
    "CatBoost":              CatBoostClassifier(iterations=500, random_state=SEED, verbose=False),
}

results = []
print(f"{'Model':<25} {'CV Acc (mean)':<16} {'CV Acc (±std)':<16} {'Val Acc'}")
print("─" * 75)

for name, model in MODELS.items():
    cv_scores = cross_val_score(model, X_train_pp, y_train,
                                cv=cv, scoring="accuracy", n_jobs=-1)
    model.fit(X_train_pp, y_train)
    val_acc = accuracy_score(y_val, model.predict(X_val_pp))
    results.append({
        "Model": name,
        "CV_Mean": cv_scores.mean(),
        "CV_Std":  cv_scores.std(),
        "Val_Acc": val_acc
    })
    print(f"{name:<25} {cv_scores.mean()*100:.3f}%{'':<9} ±{cv_scores.std()*100:.3f}%{'':<9} {val_acc*100:.3f}%")

results_df = pd.DataFrame(results).sort_values("CV_Mean", ascending=False).reset_index(drop=True)


In [ ]:
# ─── Visual comparison ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(20, 7))

# CV scores with error bars
sorted_results = results_df.sort_values("CV_Mean")
colors = ["#5B8DB8" if n not in ["LightGBM", "XGBoost", "CatBoost", "Random Forest"] 
          else "#E8825A" for n in sorted_results["Model"]]
bars = axes[0].barh(sorted_results["Model"], sorted_results["CV_Mean"] * 100,
                    xerr=sorted_results["CV_Std"] * 100,
                    color=colors, edgecolor="white", linewidth=0.8,
                    capsize=4, height=0.6)
axes[0].set_title("5-Fold CV Accuracy (mean ± std)", fontweight="bold", fontsize=13)
axes[0].set_xlabel("Accuracy (%)")
axes[0].axvline(x=sorted_results["CV_Mean"].max() * 100, color="red",
                linestyle="--", alpha=0.5, linewidth=1)
for bar, val in zip(bars, sorted_results["CV_Mean"] * 100):
    axes[0].text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                 f"{val:.2f}%", va="center", fontsize=9)

# CV vs Val scatter
axes[1].scatter(results_df["CV_Mean"] * 100, results_df["Val_Acc"] * 100,
                s=120, alpha=0.8, color="#5B8DB8", edgecolors="white", linewidth=1.5)
for _, row in results_df.iterrows():
    axes[1].annotate(row["Model"], (row["CV_Mean"]*100, row["Val_Acc"]*100),
                     textcoords="offset points", xytext=(6, 0), fontsize=8)
lims = [min(results_df["CV_Mean"].min(), results_df["Val_Acc"].min()) * 100 - 1,
        max(results_df["CV_Mean"].max(), results_df["Val_Acc"].max()) * 100 + 1]
axes[1].plot(lims, lims, "r--", alpha=0.4, linewidth=1, label="CV = Val")
axes[1].set_xlabel("CV Accuracy (%)")
axes[1].set_ylabel("Validation Accuracy (%)")
axes[1].set_title("CV vs Validation Accuracy\n(points above diagonal = generalising well)", 
                   fontweight="bold", fontsize=13)
axes[1].legend()

plt.tight_layout()
plt.show()
print("\nTop 3 models by CV score:")
print(results_df[["Model", "CV_Mean", "Val_Acc"]].head(3).to_string(index=False))


## 🔬 Hyperparameter Optimisation with Optuna

We tune the top-performing gradient boosting models using Optuna's Tree-structured Parzen Estimator (TPE) sampler with early stopping.


In [ ]:
# ─── Tune LightGBM ───────────────────────────────────────────────────────────
def lgbm_objective(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 200, 1000),
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves":        trial.suggest_int("num_leaves", 20, 150),
        "max_depth":         trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 60),
        "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "random_state": SEED, "verbose": -1
    }
    model = LGBMClassifier(**params)
    scores = cross_val_score(model, X_train_pp, y_train,
                             cv=StratifiedKFold(3, shuffle=True, random_state=SEED),
                             scoring="accuracy", n_jobs=-1)
    return scores.mean()

lgbm_study = optuna.create_study(direction="maximize",
                                  sampler=optuna.samplers.TPESampler(seed=SEED))
lgbm_study.optimize(lgbm_objective, n_trials=50, show_progress_bar=True)

print(f"\nLGBM best CV accuracy : {lgbm_study.best_value*100:.4f}%")
print("Best params:", lgbm_study.best_params)


In [ ]:
# ─── Tune XGBoost ────────────────────────────────────────────────────────────
def xgb_objective(trial):
    params = {
        "n_estimators":     trial.suggest_int("n_estimators", 200, 1000),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth":        trial.suggest_int("max_depth", 3, 10),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma":            trial.suggest_float("gamma", 0, 5),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "use_label_encoder": False, "eval_metric": "logloss",
        "verbosity": 0, "random_state": SEED
    }
    model = XGBClassifier(**params)
    scores = cross_val_score(model, X_train_pp, y_train,
                             cv=StratifiedKFold(3, shuffle=True, random_state=SEED),
                             scoring="accuracy", n_jobs=-1)
    return scores.mean()

xgb_study = optuna.create_study(direction="maximize",
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
xgb_study.optimize(xgb_objective, n_trials=50, show_progress_bar=True)

print(f"\nXGB best CV accuracy : {xgb_study.best_value*100:.4f}%")
print("Best params:", xgb_study.best_params)


In [ ]:
# ─── Build tuned models ───────────────────────────────────────────────────────
lgbm_tuned = LGBMClassifier(**lgbm_study.best_params, random_state=SEED, verbose=-1)
xgb_tuned  = XGBClassifier(**xgb_study.best_params, use_label_encoder=False,
                            eval_metric="logloss", verbosity=0, random_state=SEED)
cat_tuned   = CatBoostClassifier(iterations=600, learning_rate=0.05,
                                  depth=7, random_state=SEED, verbose=False)
rf_tuned    = RandomForestClassifier(n_estimators=500, max_depth=None,
                                      min_samples_leaf=2, random_state=SEED, n_jobs=-1)

# Fit all tuned models
for name, m in [("LightGBM", lgbm_tuned), ("XGBoost", xgb_tuned),
                ("CatBoost", cat_tuned), ("RandomForest", rf_tuned)]:
    m.fit(X_train_pp, y_train)
    val_acc = accuracy_score(y_val, m.predict(X_val_pp))
    val_auc = roc_auc_score(y_val, m.predict_proba(X_val_pp)[:, 1])
    print(f"{name:<15}  Val Acc: {val_acc*100:.4f}%   Val AUC: {val_auc:.4f}")


## 🧩 Stacking Ensemble — Final Model

A stacking ensemble uses the tuned base models as level-0 learners with a Logistic Regression meta-learner trained on their out-of-fold predictions.


In [ ]:
stacking_model = StackingClassifier(
    estimators=[
        ("lgbm", lgbm_tuned),
        ("xgb",  xgb_tuned),
        ("cat",  cat_tuned),
        ("rf",   rf_tuned),
    ],
    final_estimator=LogisticRegression(max_iter=1000, C=1.0),
    cv=5,
    passthrough=False,
    n_jobs=-1
)

stacking_model.fit(X_train_pp, y_train)

# Evaluate
y_val_pred  = stacking_model.predict(X_val_pp)
y_val_proba = stacking_model.predict_proba(X_val_pp)[:, 1]

val_acc = accuracy_score(y_val, y_val_pred)
val_f1  = f1_score(y_val, y_val_pred)
val_auc = roc_auc_score(y_val, y_val_proba)

print(f"Stacking Ensemble — Validation Results")
print(f"{'─'*40}")
print(f"  Accuracy  : {val_acc*100:.4f}%")
print(f"  F1 Score  : {val_f1:.4f}")
print(f"  ROC-AUC   : {val_auc:.4f}")
print()
print(classification_report(y_val, y_val_pred, target_names=["Not Transported", "Transported"]))


In [ ]:
# ─── Confusion matrix + ROC curve ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Confusion matrix
cm = confusion_matrix(y_val, y_val_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["Not Transported", "Transported"],
            yticklabels=["Not Transported", "Transported"],
            linewidths=0.5)
axes[0].set_title("Confusion Matrix\nStacking Ensemble", fontweight="bold", fontsize=13)
axes[0].set_ylabel("Actual")
axes[0].set_xlabel("Predicted")

# Normalised confusion matrix
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt=".2%", cmap="Blues", ax=axes[1],
            xticklabels=["Not Transported", "Transported"],
            yticklabels=["Not Transported", "Transported"],
            linewidths=0.5)
axes[1].set_title("Normalised Confusion Matrix", fontweight="bold", fontsize=13)
axes[1].set_ylabel("Actual")
axes[1].set_xlabel("Predicted")

# ROC curve
fpr, tpr, _ = roc_curve(y_val, y_val_proba)
axes[2].plot(fpr, tpr, color=PALETTE[1], lw=2, label=f"Stacking (AUC = {val_auc:.4f})")
axes[2].plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
axes[2].fill_between(fpr, tpr, alpha=0.1, color=PALETTE[1])
axes[2].set_xlabel("False Positive Rate")
axes[2].set_ylabel("True Positive Rate")
axes[2].set_title("ROC Curve", fontweight="bold", fontsize=13)
axes[2].legend(loc="lower right")

plt.tight_layout()
plt.show()


## 📈 Learning Curves — Model Generalisation Analysis

In [ ]:
# Use LightGBM for learning curves (faster than full stack)
train_sizes, train_scores, val_scores = learning_curve(
    lgbm_tuned, X_train_pp, y_train,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
    scoring="accuracy", n_jobs=-1
)

train_mean = train_scores.mean(axis=1) * 100
train_std  = train_scores.std(axis=1) * 100
val_mean   = val_scores.mean(axis=1) * 100
val_std    = val_scores.std(axis=1) * 100

fig, ax = plt.subplots(figsize=(12, 6))
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                alpha=0.15, color=PALETTE[0])
ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std,
                alpha=0.15, color=PALETTE[1])
ax.plot(train_sizes, train_mean, "o-", color=PALETTE[0], label="Training Score", lw=2)
ax.plot(train_sizes, val_mean,   "s-", color=PALETTE[1], label="CV Validation Score", lw=2)

gap = (train_mean[-1] - val_mean[-1])
ax.axhline(y=val_mean[-1], color="gray", linestyle="--", alpha=0.5)
ax.text(train_sizes[-1] * 0.5, val_mean[-1] + 0.3, 
        f"Final gap: {gap:.2f}% (bias-variance balance)", fontsize=11, color="gray")

ax.set_xlabel("Training Examples", fontsize=12)
ax.set_ylabel("Accuracy (%)", fontsize=12)
ax.set_title("Learning Curves — LightGBM (Tuned)\nDoes more data help?", 
             fontweight="bold", fontsize=14)
ax.legend(fontsize=12)
ax.set_ylim([75, 100])
plt.tight_layout()
plt.show()

if gap < 3:
    print("✅ Healthy gap: model generalises well. Low variance.")
elif gap < 8:
    print("⚠️  Moderate gap: slight overfitting. Could benefit from regularisation or more data.")
else:
    print("❌ Large gap: significant overfitting. Increase regularisation or collect more data.")


## 📌 Feature Importance Analysis

In [ ]:
# Get feature names after ColumnTransformer
try:
    ohe_feature_names = (preprocessor.named_transformers_["ohe"]
                         .named_steps["ohe"]
                         .get_feature_names_out(OHE_COLS).tolist())
except:
    ohe_feature_names = [f"ohe_{i}" for i in range(50)]

feature_names = (NUM_COLS + BOOL_COLS + ohe_feature_names + ORD_COLS)
# Trim to actual size
n_features = X_train_pp.shape[1]
feature_names = feature_names[:n_features]

# LightGBM native importance
importances = lgbm_tuned.feature_importances_
fi_df = (pd.DataFrame({"Feature": feature_names, "Importance": importances})
         .sort_values("Importance", ascending=False)
         .head(25))

plt.figure(figsize=(14, 8))
colors = ["#E8825A" if imp > fi_df["Importance"].quantile(0.75) 
          else "#5B8DB8" for imp in fi_df["Importance"]]
bars = plt.barh(fi_df["Feature"][::-1], fi_df["Importance"][::-1],
                color=colors[::-1], edgecolor="white", linewidth=0.5)
plt.xlabel("LightGBM Feature Importance (split-based)", fontsize=12)
plt.title("Top 25 Most Important Features\nOrange = Top Quartile", 
          fontweight="bold", fontsize=14)
plt.tight_layout()
plt.show()


## 🔍 Explainable AI — SHAP Analysis

SHAP (SHapley Additive exPlanations) provides game-theory-grounded explanations for individual predictions and global model behaviour. This section answers **why** the model makes each prediction — a critical component for trustworthy AI.


In [ ]:
# ─── Compute SHAP values (LightGBM — fastest for TreeExplainer) ──────────────
explainer   = shap.TreeExplainer(lgbm_tuned)
shap_values = explainer.shap_values(X_val_pp)

# For binary classification, shap_values is a list [class0, class1]
# Use class 1 (Transported=True) values
if isinstance(shap_values, list):
    sv = shap_values[1]
else:
    sv = shap_values

print(f"SHAP values computed for {sv.shape[0]} validation passengers.")
print(f"Feature dimensions: {sv.shape[1]}")


In [ ]:
# ─── Global feature importance — SHAP summary ────────────────────────────────
plt.figure(figsize=(14, 9))
shap.summary_plot(sv, X_val_pp, feature_names=feature_names,
                  plot_type="dot", show=False, max_display=20,
                  color_bar_label="Feature Value")
plt.title("SHAP Summary Plot — Global Feature Importance\n(Each point = one passenger; colour = feature value)",
          fontweight="bold", fontsize=13, pad=15)
plt.tight_layout()
plt.show()


In [ ]:
# ─── Bar version (mean |SHAP|) ───────────────────────────────────────────────
plt.figure(figsize=(12, 8))
shap.summary_plot(sv, X_val_pp, feature_names=feature_names,
                  plot_type="bar", show=False, max_display=20)
plt.title("Mean |SHAP| Value — Average Impact on Model Output",
          fontweight="bold", fontsize=13, pad=15)
plt.tight_layout()
plt.show()


In [ ]:
# ─── Individual passenger explanations ───────────────────────────────────────
# Pick one correctly predicted Transported and one correctly predicted Not Transported
y_val_arr    = y_val.values
y_pred_arr   = y_val_pred
correct_mask = y_val_arr == y_pred_arr

transported_idx     = np.where((y_val_arr == 1) & correct_mask)[0][0]
not_transported_idx = np.where((y_val_arr == 0) & correct_mask)[0][0]

fig, axes = plt.subplots(1, 2, figsize=(20, 6))

for ax, idx, title in [(axes[0], transported_idx, "Predicted: TRANSPORTED ✅"),
                        (axes[1], not_transported_idx, "Predicted: NOT TRANSPORTED ✅")]:
    plt.sca(ax)
    shap.waterfall_plot(
        shap.Explanation(values=sv[idx],
                         base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, list)
                                     else explainer.expected_value,
                         data=X_val_pp[idx],
                         feature_names=feature_names),
        max_display=12, show=False
    )
    ax.set_title(title, fontweight="bold", fontsize=12, pad=10)

plt.suptitle("Individual Passenger SHAP Explanations — Why did the model predict this?",
             fontweight="bold", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# ─── SHAP dependence plots for top features ──────────────────────────────────
top_features_idx = np.argsort(np.abs(sv).mean(axis=0))[::-1][:4]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, feat_idx in zip(axes.flatten(), top_features_idx):
    fname = feature_names[feat_idx]
    feature_vals = X_val_pp[:, feat_idx]
    shap_vals    = sv[:, feat_idx]
    
    sc = ax.scatter(feature_vals, shap_vals, c=feature_vals,
                    cmap="coolwarm", alpha=0.6, s=20, edgecolors="none")
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_xlabel(fname, fontsize=11)
    ax.set_ylabel("SHAP Value", fontsize=11)
    ax.set_title(f"Dependence: {fname}", fontweight="bold")
    plt.colorbar(sc, ax=ax, label="Feature value")

plt.suptitle("SHAP Dependence Plots — Top 4 Features\nHow does each feature value affect predictions?",
             fontweight="bold", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


## 🎯 Passenger Risk Profiling

Using model probabilities and SHAP values, we build a **Transportation Risk Profile** for each passenger — a concept borrowed from risk analytics that transforms our classifier into a decision-support tool.


In [ ]:
# ─── Assign risk tiers based on predicted probability ─────────────────────────
val_proba_df = pd.DataFrame({
    "Transport_Probability": y_val_proba,
    "Actual": y_val.values,
    "Predicted": y_val_pred
})
val_proba_df["Risk_Tier"] = pd.cut(
    val_proba_df["Transport_Probability"],
    bins=[0, 0.25, 0.45, 0.55, 0.75, 1.0],
    labels=["Very Low (< 25%)", "Low (25-45%)", "Uncertain (45-55%)",
            "High (55-75%)", "Very High (> 75%)"]
)

# Tier summary
tier_summary = (val_proba_df.groupby("Risk_Tier", observed=True)
                .agg(Count=("Actual", "size"),
                     Actual_Transport_Rate=("Actual", "mean"),
                     Avg_Predicted_Prob=("Transport_Probability", "mean"))
                .reset_index())
tier_summary["Actual_Transport_Rate"] = (tier_summary["Actual_Transport_Rate"] * 100).round(1)
tier_summary["Avg_Predicted_Prob"] = (tier_summary["Avg_Predicted_Prob"] * 100).round(1)
print("Risk Tier Distribution:")
print(tier_summary.to_string(index=False))


In [ ]:
# ─── Calibration analysis ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Probability distribution by actual class
for val, color, label in [(1, PALETTE[1], "Transported"), (0, PALETTE[0], "Not Transported")]:
    subset = val_proba_df[val_proba_df["Actual"] == val]["Transport_Probability"]
    axes[0].hist(subset, bins=40, alpha=0.65, color=color, label=label, edgecolor="none")
axes[0].axvline(0.5, color="red", linestyle="--", linewidth=1.5, label="Decision threshold")
axes[0].set_title("Predicted Probability Distribution\nby True Class", fontweight="bold")
axes[0].set_xlabel("P(Transported)")
axes[0].legend()

# Risk tier bar chart
tier_colors = ["#2E86AB", "#5B8DB8", "#F6C90E", "#E8825A", "#C62828"]
axes[1].bar(tier_summary["Risk_Tier"].astype(str),
            tier_summary["Actual_Transport_Rate"],
            color=tier_colors, edgecolor="white", linewidth=1)
axes[1].set_title("Actual Transport Rate by Risk Tier\n(Model Calibration Check)", fontweight="bold")
axes[1].set_xlabel("Risk Tier")
axes[1].set_ylabel("Actual Transport Rate (%)")
axes[1].tick_params(axis="x", rotation=20)

# Threshold sensitivity
thresholds = np.linspace(0.3, 0.7, 50)
precisions, recalls, f1s = [], [], []
for t in thresholds:
    preds_t = (y_val_proba >= t).astype(int)
    precisions.append(precision_score(y_val, preds_t, zero_division=0))
    recalls.append(recall_score(y_val, preds_t, zero_division=0))
    f1s.append(f1_score(y_val, preds_t, zero_division=0))

axes[2].plot(thresholds, precisions, label="Precision", color="#5B8DB8", lw=2)
axes[2].plot(thresholds, recalls,    label="Recall",    color="#E8825A", lw=2)
axes[2].plot(thresholds, f1s,        label="F1 Score",  color="#2ECC71", lw=2, linestyle="--")
best_t = thresholds[np.argmax(f1s)]
axes[2].axvline(best_t, color="red", linestyle=":", linewidth=1.5,
                label=f"Best threshold (F1): {best_t:.2f}")
axes[2].axvline(0.5, color="gray", linestyle="--", alpha=0.5, label="Default: 0.50")
axes[2].set_xlabel("Decision Threshold")
axes[2].set_ylabel("Score")
axes[2].set_title("Precision–Recall–F1 vs Threshold\n(Search-and-Rescue Sensitivity Analysis)", fontweight="bold")
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()
print(f"Optimal F1 threshold: {best_t:.2f}  |  F1 at optimal: {max(f1s):.4f}")


## 🕵️ Anomaly Detection — Curious Passengers

Some passengers exhibit contradictory behaviour patterns. We flag these anomalies — passengers who violate the expected rules of the voyage.


In [ ]:
# Merge predictions back to validation slice for analysis
val_indices = X_val.index
val_analysis = train_fe.loc[val_indices].copy()
val_analysis["Transport_Prob"] = y_val_proba
val_analysis["Predicted"]      = y_val_pred
val_analysis["Correct"]        = (y_val_pred == y_val.values).astype(int)

# Anomaly 1: CryoSleep passengers who spent money (should be impossible — confined to cabin)
cryo_spenders = val_analysis[
    (val_analysis["CryoSleep"].isin([True, "True", 1])) &
    (val_analysis["Total_Expenditure"] > 0)
]
print(f"🚨 CryoSleep passengers with non-zero spending: {len(cryo_spenders)}")
if len(cryo_spenders):
    print(f"   Transport rate among these anomalies: {cryo_spenders['Transported'].astype(int).mean()*100:.1f}%")

# Anomaly 2: VIP passengers who spent nothing
vip_no_spend = val_analysis[
    (val_analysis["VIP"].isin([True, "True", 1])) &
    (val_analysis["No_Spending"] == 1)
]
print(f"🚨 VIP passengers with zero spending: {len(vip_no_spend)}")
if len(vip_no_spend):
    print(f"   Transport rate among these anomalies: {vip_no_spend['Transported'].astype(int).mean()*100:.1f}%")

# Anomaly 3: Uncertain predictions (probability between 0.4 and 0.6)
uncertain = val_analysis[
    (val_analysis["Transport_Prob"] >= 0.4) &
    (val_analysis["Transport_Prob"] <= 0.6)
]
print(f"\n⚠️  Passengers in uncertain zone (0.4–0.6 probability): {len(uncertain)}")
print(f"   Correct prediction rate for these: {uncertain['Correct'].mean()*100:.1f}%")
print(f"   This uncertainty zone has lower accuracy — as expected.")


## 🚀 Final Predictions & Submission

In [ ]:
# Generate predictions using optimal threshold
test_proba      = stacking_model.predict_proba(X_test_pp)[:, 1]
test_predictions = (test_proba >= best_t).astype(bool)

submission = pd.DataFrame({
    "PassengerId": submission_ids.values,
    "Transported": test_predictions
})

print(f"Submission shape: {submission.shape}")
print(f"\nPrediction distribution:")
print(f"  Transported (True):     {submission['Transported'].sum():,}  ({submission['Transported'].mean()*100:.1f}%)")
print(f"  Not Transported (False):{(~submission['Transported']).sum():,}  ({(~submission['Transported']).mean()*100:.1f}%)")
submission.head(10)


In [ ]:
submission.to_csv("spaceship_titanic_submission.csv", index=False)
print("✅ Submission saved: spaceship_titanic_submission.csv")


## 📋 Project Summary & Key Insights

### Model Performance

| Metric | Value |
|---|---|
| Validation Accuracy | See `val_acc` |
| Validation F1 | See `val_f1` |
| Validation ROC-AUC | See `val_auc` |
| Optimisation | Optuna TPE (50 trials × 2 models) |
| Final Model | Stacking (LGBM + XGB + CatBoost + RF) |

### Key Findings

1. **CryoSleep is the strongest predictor** — passengers in suspended animation showed dramatically different transport rates, and their zero-spending pattern creates a near-perfect signal.

2. **Cabin location matters** — Deck and cabin region carry significant information. Passengers on certain decks were transported at rates far above average.

3. **Spending behaviour is a proxy for passenger type** — Zero spenders and high spenders have very different transport fates, suggesting the anomaly may have interacted with lifestyle patterns aboard the ship.

4. **Travel group dynamics add signal** — Solo travellers and group leaders show distinct transport rates, suggesting social/spatial clustering effects.

5. **Route selection correlates with fate** — The HomePlanet × Destination heatmap reveals that certain interplanetary routes carried higher transport risk, potentially reflecting which ship sections passengers occupied.

### Methodology Highlights

- ✅ Leak-free sklearn Pipeline (fit on train, transform on test)
- ✅ Optuna hyperparameter optimisation (TPE sampler)
- ✅ 5-fold stratified cross-validation for reliable estimates
- ✅ SHAP explainability (global + individual passenger level)
- ✅ Threshold optimisation via F1 maximisation
- ✅ Risk tier profiling for decision-support framing
- ✅ Anomaly detection for data quality insights
- ✅ Comprehensive behavioral analytics and segmentation


---
*This project was developed as an original portfolio piece combining standard competition ML with explainable AI, behavioral analytics, and risk profiling frameworks. The methodology goes beyond accuracy optimisation toward building trustworthy, interpretable models.*
